# Exercises (Student) - MCP Client with LLM

In [7]:
%pip install -q "mcp[cli]>=1.0,<2.0" nest_asyncio requests

In [8]:

import os
from pathlib import Path
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = False  # flip True if GITHUB_TOKEN is set


In [9]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


In [10]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@mcp.tool()
def greet(name: str) -> str:
    """Return a greeting string."""
    return f"Hello, {name}!"

@mcp.resource("greeting://sample")
def greeting_resource() -> str:
    """Return a sample greeting as an MCP resource."""
    return "Hello from DemoServer!"

if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting server.py


## Exercise 1 (provide answer)

**Answer:** STDIO is simpler for local MCP development because the client starts the server as a subprocess and exchanges JSON-RPC messages through its standard input and output. It does not require an HTTP server, open port, URL, or network configuration. It is convenient for local testing, although HTTP is more suitable for remote clients and networked deployments.

## Exercise 2

In [11]:
import nest_asyncio
import subprocess
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)
    async with stdio_client(params, errlog=subprocess.DEVNULL) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()

In [12]:
# In a new cell
await ex2_connect()
print("Exercise 2: OK (connected and initialized)")


Exercise 2: OK (connected and initialized)


## Exercise 3

In [13]:
import subprocess
async def ex3_list():
    params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)
    async with stdio_client(params, errlog=subprocess.DEVNULL) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            resources = await session.list_resources()
            print("RESOURCES:")
            for resource in resources.resources:
                print("-", resource.uri)
            tools = await session.list_tools()
            print("TOOLS:")
            for tool in tools.tools:
                print("-", tool.name, tool.inputSchema.get("properties", {}))

In [14]:
await ex3_list()

RESOURCES:
- greeting://sample
TOOLS:
- add {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
- greet {'name': {'title': 'Name', 'type': 'string'}}


## Exercise 4

**Answer:** The MCP server exposes tools with names, descriptions, and JSON input schemas. The client discovers these definitions using `session.list_tools()`. `convert_to_llm_tool()` maps each MCP tool into an LLM-compatible function specification: the tool name becomes `function.name`, its description becomes `function.description`, and its input schema supplies `parameters`. The LLM (or stub planner) selects a function and arguments; the client then invokes the original MCP tool with `session.call_tool()`. The conversion occurs in the **client**, not in the MCP server.

In [15]:
def convert_to_llm_tool(tool):
    """Convert a discovered MCP tool into an LLM function specification."""
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "MCP tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }

## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like “Add 2 to 20.”

In [16]:
import json
import os
import re
import subprocess
from typing import Any, Dict, List


def stub_plan(prompt: str, functions: List[Dict[str, Any]]):
    """Plan a deterministic add tool call without contacting an LLM."""
    available = {item["function"]["name"] for item in functions}
    match = re.search(r"add\s+(-?\d+)\s+to\s+(-?\d+)", prompt, re.IGNORECASE)
    if match and "add" in available:
        return [{"name": "add", "args": {"a": int(match.group(1)), "b": int(match.group(2))}}]
    return []


def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    """Return planned MCP calls from a stub or optional real LLM."""
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient(
        endpoint="https://models.inference.ai.azure.com",
        credential=AzureKeyCredential(token),
    )
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},
                  {"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    for tc in resp.choices[0].message.tool_calls or []:
        args = tc.function.arguments
        calls.append({"name": tc.function.name,
                      "args": json.loads(args) if isinstance(args, str) else args})
    return calls

In [17]:
async def ex5_run(prompt: str = "Add 2 to 20"):
    params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)
    async with stdio_client(params, errlog=subprocess.DEVNULL) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = await session.list_tools()
            functions = [convert_to_llm_tool(tool) for tool in tools.tools]
            print("LLM function specifications:", json.dumps(functions, indent=2))
            calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
            print("tool_calls:", calls)
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                print("result:", [getattr(c, "text", str(c)) for c in result.content])

In [18]:
await ex5_run("Add 2 to 20")

LLM function specifications: [
  {
    "type": "function",
    "function": {
      "name": "add",
      "description": "Add two numbers.",
      "parameters": {
        "type": "object",
        "properties": {
          "a": {
            "title": "A",
            "type": "integer"
          },
          "b": {
            "title": "B",
            "type": "integer"
          }
        },
        "required": [
          "a",
          "b"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "greet",
      "description": "Return a greeting string.",
      "parameters": {
        "type": "object",
        "properties": {
          "name": {
            "title": "Name",
            "type": "string"
          }
        },
        "required": [
          "name"
        ]
      }
    }
  }
]
tool_calls: [{'name': 'add', 'args': {'a': 2, 'b': 20}}]
result: ['22']


## Optional - add multiply(a, b) and rerun